## Blood Work Analysis Pipeline

A two-stage LLM pipeline:
- **Stage 1:** Extract and flag abnormal values from a blood report
- **Stage 2:** Generate a health summary and Indian diet plan based on flagged values

In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
# from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0.7)

## Sample Blood Report

In [1]:
with open("blood_work.txt", "r") as f:
    blood_report = f.read()

print(blood_report[:200])

Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
--------------------------
Hemoglobin:        15.1 g/dL        (Normal: 13.5–17.5)
Hematocrit:        44%            


## Stage 1 - Extract and Flag Abnormal Values

In [4]:
extraction_prompt = f"""
You are a medical data extraction assistant.

From the blood report below, extract ALL test values and classify each one as HIGH, LOW or NORMAL
based on the reference ranges provided in the report.

Format your response as:
- Test Name: value | Status: HIGH/LOW/NORMAL | Reference: range

Blood Report:
{blood_report}
"""

extraction_response = llm.invoke(extraction_prompt)
extracted_values = extraction_response.text

print("=== STAGE 1: EXTRACTED VALUES ===")
print(extracted_values)


=== STAGE 1: EXTRACTED VALUES ===
- Hemoglobin: 15.1 g/dL | Status: NORMAL | Reference: 13.5–17.5  
- Hematocrit: 44 % | Status: NORMAL | Reference: 41–53%  
- WBC: 6.8 x10^3/uL | Status: NORMAL | Reference: 4.5–11.0  
- Platelets: 220 x10^3/uL | Status: NORMAL | Reference: 150–400  

- Total Cholesterol: 238 mg/dL | Status: HIGH | Reference: <200  
- LDL Cholesterol: 162 mg/dL | Status: HIGH | Reference: <100  
- HDL Cholesterol: 36 mg/dL | Status: LOW | Reference: >40  
- Triglycerides: 188 mg/dL | Status: HIGH | Reference: <150  

- Glucose (Fasting): 92 mg/dL | Status: NORMAL | Reference: 70–99  
- HbA1c: 5.3 % | Status: NORMAL | Reference: <5.7%  
- Creatinine: 1.0 mg/dL | Status: NORMAL | Reference: 0.7–1.3  
- eGFR: 82 mL/min | Status: NORMAL | Reference: >60  

- ALT: 28 U/L | Status: NORMAL | Reference: 7–40  
- AST: 25 U/L | Status: NORMAL | Reference: 10–40  
- Bilirubin Total: 0.8 mg/dL | Status: NORMAL | Reference: 0.2–1.2


## Stage 2 - Health Summary and Indian Diet Plan

In [9]:
diet_prompt = f"""
You are a clinical nutritionist specializing in Indian dietary habits.

Based on the blood work analysis below, write:
1. A short health summary in 4-5 lines explaining the patient's condition in simple language.
2. A short, practical Indian diet plan having only two sections (1) Food to avoid (2) Food to eat more of.
   Do not include any other sections in diet plan.

Blood Work Analysis:
{extracted_values}
"""

diet_response = llm.invoke(diet_prompt)

print("=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
print(diet_response.text)

=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===
**Health Summary**  
- Your blood counts (Hb, WBC, platelets) and kidney‑liver function are all normal.  
- Fasting glucose and HbA1c are within healthy limits, so diabetes is not a concern.  
- Lipid profile shows high total cholesterol, high LDL, low HDL, and elevated triglycerides – a classic “bad‑cholesterol” pattern.  
- This dyslipidaemia raises the risk of heart disease, and the primary way to improve it is through diet and regular physical activity.  

---

### Food to avoid  
- Deep‑fried foods (pakoras, samosas, bhajis) and anything cooked in a lot of oil or ghee  
- Full‑fat dairy (whole milk, cream, butter, full‑fat paneer)  
- Red meat and processed meats (mutton, beef, sausages, kebabs)  
- Refined grains and sweets (white rice, maida/maize flour rotis, white bread, sweets, sugary drinks, packaged snacks)  
- Trans‑fat sources (store‑bought biscuits, cookies, pastries, margarine)  
- Excess coconut or palm oil and heavy gravies 